# 13v5. Adaptive entity selector with validation-based early stopping

This notebook evaluates a redesigned **fusion + adaptive constraint**
architecture on the repaired WebNLG protocol.

## Architecture

```text
Input graph
    ↓
Frozen R-GCN + graph fusion + BART
    ↓
Decoder-conditioned entity selector
    ↓
NONE or one graph entity
    ↓
Exact selected-entity completion
    ↓
Optional generic Hard-v2 backstop
```

Only the selector is trained. The fusion checkpoint, BART, and R-GCN remain
frozen.

## What v5 changes

- Trains for at most **6 epochs**.
- Validates after every epoch.
- Uses **patience = 2**.
- Saves both resume and best checkpoints.
- Restores the best checkpoint before generation evaluation.
- Selects checkpoints by entity-start F1, with identity accuracy as tie-breaker.
- Uses a new selector-validation/tuning/confirmation partition that excludes:
  - the historical seed-7 development subset;
  - the historical seed-21 development subset;
  - the already-inspected v4 Random(33) tuning subset.
- Invalidates stale tuning results whenever the best selector checkpoint changes.
- Keeps the verified 2×2 comparison:
  1. fusion off;
  2. fusion + adaptive selector commitment;
  3. fusion + Hard-v2;
  4. fusion + adaptive selector commitment + Hard-v2.

## Development split discipline

The 1,667-example development set is partitioned as follows:

- 300 historical seed-7 examples: excluded;
- 500 historical seed-21 examples: excluded;
- 300 previously inspected v4 tuning examples: excluded;
- 167 examples: selector checkpoint validation;
- 200 examples: new generation tuning;
- 200 examples: untouched generation confirmation.

The repaired test set is decoded only when the selector-only arm passes both
new development screens.

## Important

Use this notebook in a fresh Colab runtime. It writes to
`selector_eval_v5_earlystop` and does **not** reuse the old v4 selector or its
failed tuning configuration. Resume is supported within v5.

In [ ]:
# 1. Bootstrap (Xet disabled BEFORE any HF import — lesson from 12b).
import os
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '600'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

!pip -q install "transformers>=4.44,<5" "huggingface_hub>=0.25,<1" nltk rouge-score accelerate sentencepiece sacremoses sacrebleu

import sys, json, pickle, random, time, shutil, subprocess, re, unicodedata, hashlib
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from google.colab import drive
drive.mount('/content/drive')
print('Torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU runtime required'

In [ ]:
# 1.1 PyTorch Geometric.
try:
    import torch_geometric
    print('ok:', torch_geometric.__version__)
except Exception:
    tv = torch.__version__.split('+')[0]; cv = torch.version.cuda
    url = f'https://data.pyg.org/whl/torch-{tv}+cu{cv.replace(".", "")}.html'
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch-geometric', 'torch-scatter', 'torch-sparse', '-f', url])
    import torch_geometric

In [ ]:
# 2. Paths, module, local BART snapshot (curl, resumable — no Hub client for weights).
PROJECT_DIR = '/content/drive/MyDrive/kg_llm_project'
PROCESSED_DIR = f'{PROJECT_DIR}/baseline-bart-webnlg/processed'
CKPT_FUSION = f'{PROJECT_DIR}/fusion_only_outputs/checkpoints/fusion_only/model_best.pt'
HV2_EVAL = f'{PROJECT_DIR}/hard_v2_eval_fixed_v2'      # preferred legacy location
ALLOW_REFERENCE_REGEN = True  # fair fallback when historical files are absent

# Selector-training controls.
MAX_SELECTOR_EPOCHS = 6
PATIENCE = 2
MIN_DELTA = 1e-4
SELECTOR_BATCH_SIZE = 24
SELECTOR_LR = 1e-3
NONE_WEIGHT = 0.3
EVAL_OUT = f'{PROJECT_DIR}/selector_eval_v5_earlystop'
os.makedirs(EVAL_OUT, exist_ok=True)
for p in (f'{PROJECT_DIR}/fixed_ablation_common.py', PROCESSED_DIR, CKPT_FUSION):
    assert os.path.exists(p), f'missing: {p}'
shutil.copy(f'{PROJECT_DIR}/fixed_ablation_common.py', '/content/fixed_ablation_common.py')
if '/content' not in sys.path: sys.path.insert(0, '/content')
import fixed_ablation_common as fac
from fixed_ablation_common import (load_artifacts, FusionOnlyGNNModel, load_variant_checkpoint_resume,
                                   add_final_logits_bias, pad_kg_nodes, find_entity_token_spans)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE = 'cuda'
MAX_GEN_LEN, MAX_INPUT_LEN, MAX_TARGET_LEN = 128, 256, 128
LOCK_MIN_DEPTH, ESCAPE_MARGIN = 2, 10.0

from urllib.parse import quote
BART_LOCAL = '/content/bart-base-local'; os.makedirs(BART_LOCAL, exist_ok=True)
FILES = {'config.json': 1000, 'vocab.json': 800000, 'merges.txt': 400000,
         'tokenizer.json': 1000000, 'model.safetensors': 500000000}
for fn, mn in FILES.items():
    dest = os.path.join(BART_LOCAL, fn)
    if os.path.isfile(dest) and os.path.getsize(dest) >= mn:
        continue
    url = f'https://huggingface.co/facebook/bart-base/resolve/main/{quote(fn)}?download=true'
    rc = subprocess.run(['curl', '-L', '--fail', '--retry', '12', '--retry-delay', '5', '--retry-all-errors',
                         '--connect-timeout', '30', '--speed-time', '90', '--speed-limit', '1024',
                         '-C', '-', '-o', dest + '.part', url]).returncode
    if rc != 0:
        if os.path.exists(dest + '.part'): os.remove(dest + '.part')
        rc = subprocess.run(['curl', '-L', '--fail', '-o', dest + '.part', url]).returncode
    assert rc == 0 and os.path.getsize(dest + '.part') >= mn, f'download failed: {fn}'
    os.replace(dest + '.part', dest)
    print('downloaded', fn)
print('BART snapshot ready')

In [ ]:
# 3. Load data + frozen fusion model.
from transformers import BartTokenizer, BartForConditionalGeneration
tokenizer = BartTokenizer.from_pretrained(BART_LOCAL)
bart = BartForConditionalGeneration.from_pretrained(BART_LOCAL, local_files_only=True).to(DEVICE)
data, graphs, vocab = load_artifacts(PROCESSED_DIR)
num_relations = len(vocab['relation_vocab']) * 2
model = FusionOnlyGNNModel(bart, num_relations=num_relations).to(DEVICE)
model = load_variant_checkpoint_resume(model, CKPT_FUSION, DEVICE, strict=False)
model.eval()
for p in model.parameters(): p.requires_grad_(False)
print('frozen fusion model ready | splits:', {k: len(v) for k, v in data.items()})

In [ ]:
# 4. Manifest + metrics + TrieMapV2 + hard mask (ports of the 12b-verified cells).
from collections import OrderedDict
from difflib import SequenceMatcher
import sacrebleu

def _tp(t):
    if isinstance(t, dict): return str(t.get('subject','')), str(t.get('predicate','')), str(t.get('object',''))
    return str(t[0]), str(t[1]), str(t[2])

groups = OrderedDict()
for i, ex in enumerate(data['test']):
    key = json.dumps([_tp(t) for t in ex['triples']], ensure_ascii=False)
    g = groups.setdefault(key, {'first_index': i, 'references': []})
    for v in list(ex.get('all_targets') or []) + [ex.get('target','')]:
        v = str(v).strip()
        if v and v not in g['references']: g['references'].append(v)
train_predicates = {_tp(t)[1] for ex in data['train'] for t in ex['triples']}
ITEMS = []
for uid, (key, g) in enumerate(groups.items()):
    ex = dict(data['test'][g['first_index']]); ex['all_targets'] = g['references']
    ITEMS.append({'uid': uid, 'idx': g['first_index'], 'ex': ex,
                  'unseen': bool({_tp(t)[1] for t in ex['triples']} - train_predicates)})
assert len(ITEMS) == 2510 and sum(x['unseen'] for x in ITEMS) == 752

_TR = {'ø':'o','Ø':'o','æ':'ae','Æ':'ae','œ':'oe','Œ':'oe','ð':'d','Ð':'d','þ':'th','Þ':'th',
       'ł':'l','Ł':'l','ß':'ss','đ':'d','Đ':'d','ħ':'h','ı':'i','İ':'i','ŋ':'ng'}
_MONTHS = ['january','february','march','april','may','june','july','august','september','october','november','december']
_DET = re.compile(r'^(the|a|an)\s+')
_STOP = frozenset(('the a an and or but if then this that these those it its he she they them his her their '
                   'in on at of to by with for from as is are was were be been being there here also however').split())
def _sa(s):
    s = ''.join(_TR.get(c, c) for c in str(s))
    return ''.join(c for c in unicodedata.normalize('NFKD', s) if not unicodedata.combining(c))
def _norm(s):
    s = _sa(str(s)).lower().strip().strip('"').strip("'")
    s = re.sub(r'[^a-z0-9 ]', ' ', s); s = re.sub(r'\s+', ' ', s).strip()
    return _DET.sub('', s)
def _wm(t, s):
    return bool(s) and re.search(r'(?<![a-z0-9])' + re.escape(s) + r'(?![a-z0-9])', t) is not None
def _dst(v):
    toks = set(); raw = _sa(str(v)).strip().strip('"').strip("'")
    m = re.match(r'^(\d{3,4})-(\d{1,2})-(\d{1,2})$', raw)
    if m:
        y, mo, d = map(int, m.groups()); toks.add(str(y))
        if 1 <= mo <= 12: toks.add(_MONTHS[mo-1]); toks.add(_MONTHS[mo-1][:3])
        toks.update({str(d), str(d).zfill(2)}); toks.update({f'{d}{sx}' for sx in ('st','nd','rd','th')})
    elif re.match(r'^\d{3,4}$', raw): toks.add(raw)
    return toks
def _dg(s): return re.sub(r'[^0-9]', '', str(s))
def grounding_score(prediction, triples):
    pn = _norm(prediction)
    forms, tokens, num = [], set(), set()
    for t in triples:
        s, p, o = _tp(t)
        for v in (s, o):
            n = _norm(v)
            if n: forms.append(n); tokens.update(n.split())
            tokens.update(_dst(v)); d = _dg(v)
            if d: num.add(d)
        tokens.update(_norm(re.sub(r'([a-z])([A-Z])', r'\1 \2', p)).split())
    fd = [f.replace(' ', '') for f in forms]
    found = sum(1 for f in forms if _wm(pn, f) or all(_wm(pn, w) for w in f.split()))
    recall = found / len(forms) if forms else 1.0
    men = set()
    for m in re.findall(r'[A-Z][A-Za-z]*(?:[ -][A-Z][A-Za-z]*)*', _sa(prediction)):
        n = _norm(m)
        if len(n) >= 3 and (' ' in n or n not in _STOP): men.add(n)
    for m in re.findall(r'[A-Za-z0-9]+(?:[./\-][A-Za-z0-9]+)*', str(prediction)):
        if any(c.isdigit() for c in m): men.add(_norm(m))
    men = {m for m in men if m}
    def ok(m):
        for f in forms:
            if m == f or _wm(f, m): return True
        if all(t in tokens for t in m.split()): return True
        d = _dg(m)
        if d and any(d in c or c in d for c in num): return True
        md = m.replace(' ', '')
        return len(md) >= 3 and any(md in x or x in md for x in fd)
    hall = sorted(m for m in men if not ok(m))
    corr = [m for m in hall if max((SequenceMatcher(None, m, f).ratio() for f in forms), default=0) >= 0.55]
    return {'halluc': len(hall)/len(men) if men else 0.0, 'recall': recall,
            'hallucinated': hall, 'corruptions': corr}
ART = {'iso': r'[A-Za-z],? \d{3,4}-\d{1,2}-\d{1,2}|\d{1,2}(st|nd|rd|th)? [A-Za-z]+ \d{3,4}-\d{1,2}-\d{1,2}',
       'paren': r'\((The [^)]+album|[0-9]{4} film|film|band|song|album|actor[^)]*|footballer[^)]*|musician[^)]*)\)',
       'unit': r'\d[\d.,]*\s*\((milli|centi|kilo)?(metres|meters|grams|litres|liters|inches)\)'}
def artrow(p): return any(re.search(x, str(p)) for x in ART.values())
def corpus_bleu_lc(preds, refs_list):
    maxr = max(len(r) for r in refs_list)
    streams = [[r[k] if k < len(r) else r[0] for r in refs_list] for k in range(maxr)]
    return sacrebleu.corpus_bleu(preds, streams, lowercase=True, tokenize='13a').score

class TNode:
    __slots__ = ('ch', 'score', 'mx', 'nterm', 'terminal')
    def __init__(self):
        self.ch = {}; self.score = None; self.mx = -1e9; self.nterm = 0; self.terminal = False
_LIT = [r'^[\d\s.,:/\-+%°"]*$', r'^\d{3,4}-\d{1,2}-\d{1,2}', r'^\d+(\.\d+)?$']
def is_literal(name):
    n = str(name).strip().strip('"').strip("'").strip()
    return len(n) < 2 or any(re.match(p, n) for p in _LIT)
def clean_surface(name):
    s = str(name).strip().strip('"').strip("'").replace('_', ' ')
    s = re.sub(r'\s*\([^)]*\)', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()
class TrieMapV2:
    def __init__(self, entity_names, tokenizer):
        self.root = TNode(); self.kept = {}
        for ni, name in enumerate(entity_names):
            if is_literal(name): continue
            base = clean_surface(name)
            if not base or is_literal(base): continue
            self.kept[ni] = base
            for v in (base, ' ' + base):
                ids = tokenizer.encode(v, add_special_tokens=False)
                if not ids: continue
                n = self.root
                for tid in ids: n = n.ch.setdefault(int(tid), TNode())
                n.terminal = True
        self._cache(self.root)
    def _cache(self, n):
        nt = 1 if n.terminal else 0
        for c in n.ch.values():
            self._cache(c); nt += c.nterm
        n.nterm = nt
    def suffix_matches(self, gen_ids, lookback=20):
        out = []
        for start in range(max(0, len(gen_ids) - lookback), len(gen_ids)):
            n = self.root; okk = True
            for pos in range(start, len(gen_ids)):
                t = int(gen_ids[pos])
                if t not in n.ch: okk = False; break
                n = n.ch[t]
            if okk and n is not self.root: out.append((len(gen_ids) - start, n))
        return out
def hard_mask_v2(logits, trie, gen_ids):
    best = None
    for depth, node in trie.suffix_matches(gen_ids):
        if node.terminal or not node.ch: continue
        if (depth >= LOCK_MIN_DEPTH or node.nterm == 1) and (best is None or depth > best[0]):
            best = (depth, node)
    if best is None: return logits, False
    legal = list(best[1].ch.keys())
    if float(logits.max()) - max(float(logits[t]) for t in legal) > ESCAPE_MARGIN:
        return logits, False
    m = torch.full_like(logits, -1e9); m[legal] = 0.0
    return logits + m, True
print('protocol cells ready')

In [ ]:
# 5. Selector module + gold span labels from references.
class EntitySelector(nn.Module):
    # scores {NONE} ∪ {entities} from [h_t ; e_i ; cov_i]
    def __init__(self, d=768, hid=256):
        super().__init__()
        self.ent_mlp = nn.Sequential(nn.Linear(2*d + 1, hid), nn.ReLU(), nn.Linear(hid, 1))
        self.none_mlp = nn.Sequential(nn.Linear(d, hid), nn.ReLU(), nn.Linear(hid, 1))
    def forward(self, h, ents, cov):
        # h: (L,d) | ents: (N,d) | cov: (L,N) in {0,1}
        L, d = h.shape; N = ents.shape[0]
        he = torch.cat([h.unsqueeze(1).expand(L, N, d), ents.unsqueeze(0).expand(L, N, d),
                        cov.unsqueeze(-1)], dim=-1)
        s_ent = self.ent_mlp(he).squeeze(-1)          # (L,N)
        s_none = self.none_mlp(h)                     # (L,1)
        return torch.cat([s_none, s_ent], dim=-1)     # (L, 1+N); class 0 = NONE

def target_labels(ex, graph, target_ids):
    # label per decoder-state position t (predicting token t of target_ids):
    #  0 = NONE, i+1 = entity i STARTS at t, -100 = inside a span (ignored)
    names = list(getattr(graph, 'entity_names', []) or [])
    surfaces = [clean_surface(n) if not is_literal(n) else '' for n in names]
    lab = np.zeros(len(target_ids), dtype=np.int64)
    text = ex['target']
    spans = find_entity_token_spans(text, [s if s else '§none§' for s in surfaces], tokenizer)
    for ni, (s, e) in enumerate(spans):
        if s < 0 or not surfaces[ni]: continue
        if s < len(lab): lab[s] = ni + 1
        for k in range(s + 1, min(e, len(lab))): lab[k] = -100
    return lab
print('selector defined')

In [ ]:
# 6. Selector training with per-epoch validation and best-checkpoint restore.

SEL_LAST_PATH = os.path.join(EVAL_OUT, "selector_last_v5.pt")
SEL_BEST_PATH = os.path.join(EVAL_OUT, "selector_best_v5.pt")
SEL_HISTORY_PATH = os.path.join(
    EVAL_OUT,
    "selector_train_history_v5.json",
)
SPLIT_MANIFEST_PATH = os.path.join(
    EVAL_OUT,
    "selector_dev_splits_v5.json",
)

pad_id = tokenizer.pad_token_id
dec_start = model.bart.config.decoder_start_token_id


def atomic_json_dump(value, path, indent=2):
    temporary_path = path + ".tmp"
    with open(temporary_path, "w", encoding="utf-8") as handle:
        json.dump(
            value,
            handle,
            indent=indent,
            ensure_ascii=False,
        )
    os.replace(temporary_path, path)


def atomic_torch_save(value, path):
    temporary_path = path + ".tmp"
    torch.save(value, temporary_path)
    os.replace(temporary_path, path)


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while True:
            chunk = handle.read(1024 * 1024)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def build_v5_dev_splits():
    """Build disjoint, fixed splits from the 1,667-example dev set.

    The first three groups reproduce development subsets already used or
    inspected in earlier notebooks. They are excluded from all v5 model
    selection decisions.
    """
    universe = list(range(len(data["dev"])))
    assert len(universe) == 1667, (
        "Expected the rebuilt 1,667-example development set, "
        f"found {len(universe)}."
    )

    prior_seed7 = set(
        random.Random(7).sample(universe, 300)
    )

    pool_after_seed7 = [
        index
        for index in universe
        if index not in prior_seed7
    ]
    prior_seed21 = set(
        random.Random(21).sample(pool_after_seed7, 500)
    )

    pool_after_prior = [
        index
        for index in universe
        if index not in prior_seed7
        and index not in prior_seed21
    ]

    # This exactly reproduces the v4 Random(33) tuning subset.
    prior_v4_tune = set(
        random.Random(33).sample(pool_after_prior, 300)
    )

    unused = [
        index
        for index in universe
        if index not in prior_seed7
        and index not in prior_seed21
        and index not in prior_v4_tune
    ]
    assert len(unused) == 567

    selector_validation = set(
        random.Random(55).sample(unused, 167)
    )

    after_selector_validation = [
        index
        for index in unused
        if index not in selector_validation
    ]
    generation_tune = set(
        random.Random(66).sample(
            after_selector_validation,
            200,
        )
    )

    generation_confirm = set(
        index
        for index in after_selector_validation
        if index not in generation_tune
    )

    groups = {
        "excluded_historical_seed7": sorted(prior_seed7),
        "excluded_historical_seed21": sorted(prior_seed21),
        "excluded_inspected_v4_tune": sorted(prior_v4_tune),
        "selector_validation": sorted(selector_validation),
        "generation_tune": sorted(generation_tune),
        "generation_confirm": sorted(generation_confirm),
    }

    expected_sizes = {
        "excluded_historical_seed7": 300,
        "excluded_historical_seed21": 500,
        "excluded_inspected_v4_tune": 300,
        "selector_validation": 167,
        "generation_tune": 200,
        "generation_confirm": 200,
    }

    for name, expected_size in expected_sizes.items():
        assert len(groups[name]) == expected_size, (
            f"{name}: {len(groups[name])} != {expected_size}"
        )

    group_sets = {
        name: set(indices)
        for name, indices in groups.items()
    }
    group_names = list(group_sets)
    for first_position, first_name in enumerate(group_names):
        for second_name in group_names[first_position + 1:]:
            overlap = (
                group_sets[first_name]
                & group_sets[second_name]
            )
            assert not overlap, (
                f"Development split overlap: "
                f"{first_name} vs {second_name}: "
                f"{len(overlap)}"
            )

    all_assigned = set().union(*group_sets.values())
    assert len(all_assigned) == len(universe)
    assert all_assigned == set(universe)

    manifest = {
        "version": "v5_earlystop",
        "dev_size": len(universe),
        "seeds": {
            "historical_seed7": 7,
            "historical_seed21": 21,
            "inspected_v4_tune": 33,
            "selector_validation": 55,
            "generation_tune": 66,
            "generation_confirm": "remaining_after_seed66",
        },
        "groups": groups,
    }

    serialized = json.dumps(
        manifest,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
    manifest["sha256"] = hashlib.sha256(serialized).hexdigest()
    return manifest


if os.path.exists(SPLIT_MANIFEST_PATH):
    DEV_SPLITS = json.load(
        open(SPLIT_MANIFEST_PATH, encoding="utf-8")
    )
else:
    DEV_SPLITS = build_v5_dev_splits()
    atomic_json_dump(
        DEV_SPLITS,
        SPLIT_MANIFEST_PATH,
    )

SELECTOR_VAL_IDX = DEV_SPLITS["groups"][
    "selector_validation"
]
V5_TUNE_IDX = DEV_SPLITS["groups"][
    "generation_tune"
]
V5_CONFIRM_IDX = DEV_SPLITS["groups"][
    "generation_confirm"
]
DEV_SPLIT_SHA256 = DEV_SPLITS["sha256"]

assert not (
    set(SELECTOR_VAL_IDX) & set(V5_TUNE_IDX)
)
assert not (
    set(SELECTOR_VAL_IDX) & set(V5_CONFIRM_IDX)
)
assert not (
    set(V5_TUNE_IDX) & set(V5_CONFIRM_IDX)
)

print(
    "v5 dev splits:",
    {
        "selector_validation": len(SELECTOR_VAL_IDX),
        "generation_tune": len(V5_TUNE_IDX),
        "generation_confirm": len(V5_CONFIRM_IDX),
    },
)
print("split manifest:", SPLIT_MANIFEST_PATH)
print("split hash:", DEV_SPLIT_SHA256)


@torch.no_grad()
def teacher_states(batch_examples, batch_graphs):
    """Return frozen fused decoder states and graph entity embeddings."""
    encoder_batch = tokenizer(
        [
            example["linearized"]
            for example in batch_examples
        ],
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding=True,
        return_tensors="pt",
    ).to(DEVICE)

    target_batch = tokenizer(
        text_target=[
            example["target"]
            for example in batch_examples
        ],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding=True,
        return_tensors="pt",
    )
    labels = target_batch["input_ids"]

    decoder_input_ids = labels.clone()
    decoder_input_ids[
        decoder_input_ids == pad_id
    ] = pad_id
    decoder_input_ids = torch.cat(
        [
            torch.full(
                (decoder_input_ids.size(0), 1),
                dec_start,
                dtype=decoder_input_ids.dtype,
            ),
            decoder_input_ids[:, :-1],
        ],
        dim=1,
    ).to(DEVICE)

    from torch_geometric.data import Batch as PyGBatch

    graph_batch_object = PyGBatch.from_data_list(
        batch_graphs
    ).to(DEVICE)

    encoder_output = model.bart.model.encoder(
        input_ids=encoder_batch["input_ids"],
        attention_mask=encoder_batch[
            "attention_mask"
        ],
    )

    h_kg, _ = model.rgcn(
        graph_batch_object.x,
        graph_batch_object.edge_index,
        graph_batch_object.edge_type,
    )

    h_kg_padded, kg_mask = pad_kg_nodes(
        h_kg,
        graph_batch_object.batch,
        len(batch_examples),
    )

    decoder_output = model.bart.model.decoder(
        input_ids=decoder_input_ids,
        encoder_hidden_states=(
            encoder_output.last_hidden_state
        ),
        encoder_attention_mask=encoder_batch[
            "attention_mask"
        ],
    )

    h_fused, _, _ = model.kg_cross_attention(
        decoder_output.last_hidden_state,
        h_kg_padded,
        kg_mask,
    )

    return (
        h_fused.detach(),
        h_kg.detach(),
        graph_batch_object.batch.detach(),
        labels,
    )


def build_coverage(labels_tensor, num_entities):
    """Coverage before each target position."""
    length = int(labels_tensor.numel())
    coverage = torch.zeros(
        length,
        num_entities,
        device=DEVICE,
    )
    seen = set()

    for position in range(length):
        for entity_index in seen:
            coverage[position, entity_index] = 1.0

        value = int(labels_tensor[position].item())
        if value > 0:
            seen.add(value - 1)

    return coverage


def selector_loss_for_example(
    selector_model,
    fused_states,
    entity_embeddings,
    gold_labels,
):
    coverage = build_coverage(
        gold_labels,
        entity_embeddings.size(0),
    )

    logits = selector_model(
        fused_states.float(),
        entity_embeddings.float(),
        coverage,
    )

    class_weights = torch.ones(
        logits.size(-1),
        device=DEVICE,
    )
    class_weights[0] = NONE_WEIGHT

    return F.cross_entropy(
        logits,
        gold_labels,
        weight=class_weights,
        ignore_index=-100,
    )


@torch.no_grad()
def evaluate_selector(
    selector_model,
    indices,
    batch_size=16,
    verbose=True,
):
    """Intrinsic selector evaluation on fixed teacher-forced states."""
    selector_model.eval()

    true_positive = 0
    false_positive = 0
    false_negative = 0
    exact_identity = 0
    true_start_count = 0
    none_correct = 0
    none_count = 0
    loss_sum = 0.0
    loss_examples = 0

    for batch_start in range(
        0,
        len(indices),
        batch_size,
    ):
        batch_indices = indices[
            batch_start:
            batch_start + batch_size
        ]
        batch_examples = [
            data["dev"][index]
            for index in batch_indices
        ]
        batch_graphs = [
            graphs["dev"][index]
            for index in batch_indices
        ]

        (
            fused_states,
            all_entity_embeddings,
            graph_assignment,
            target_ids,
        ) = teacher_states(
            batch_examples,
            batch_graphs,
        )

        for local_index, (example, graph) in enumerate(
            zip(batch_examples, batch_graphs)
        ):
            entity_names = list(
                getattr(graph, "entity_names", []) or []
            )
            if not entity_names:
                continue

            node_indices = (
                graph_assignment == local_index
            ).nonzero(
                as_tuple=False
            ).flatten()

            if len(node_indices) != len(entity_names):
                raise RuntimeError(
                    "Graph node/entity-name mismatch "
                    f"for dev index "
                    f"{batch_indices[local_index]}: "
                    f"{len(node_indices)} vs "
                    f"{len(entity_names)}."
                )

            entity_embeddings = (
                all_entity_embeddings[
                    node_indices
                ].float()
            )

            target_row = target_ids[local_index]
            target_length = int(
                (
                    target_row != pad_id
                ).sum().item()
            )
            if target_length < 2:
                continue

            gold_labels = torch.from_numpy(
                target_labels(
                    example,
                    graph,
                    target_row[
                        :target_length
                    ].tolist(),
                )
            ).long().to(DEVICE)

            fused_row = fused_states[
                local_index,
                :target_length,
            ].float()

            coverage = build_coverage(
                gold_labels,
                len(entity_names),
            )
            logits = selector_model(
                fused_row,
                entity_embeddings,
                coverage,
            )

            class_weights = torch.ones(
                logits.size(-1),
                device=DEVICE,
            )
            class_weights[0] = NONE_WEIGHT

            loss = F.cross_entropy(
                logits,
                gold_labels,
                weight=class_weights,
                ignore_index=-100,
            )
            if torch.isfinite(loss):
                loss_sum += float(loss.cpu())
                loss_examples += 1

            predictions = logits.argmax(
                dim=-1
            ).detach().cpu()
            gold_cpu = gold_labels.detach().cpu()

            for position in range(target_length):
                gold_value = int(
                    gold_cpu[position].item()
                )
                if gold_value == -100:
                    continue

                predicted_value = int(
                    predictions[position].item()
                )

                if gold_value > 0:
                    true_start_count += 1
                    if predicted_value > 0:
                        true_positive += 1
                        if predicted_value == gold_value:
                            exact_identity += 1
                    else:
                        false_negative += 1
                else:
                    none_count += 1
                    if predicted_value > 0:
                        false_positive += 1
                    else:
                        none_correct += 1

    precision = (
        true_positive
        / max(true_positive + false_positive, 1)
    )
    recall = (
        true_positive
        / max(true_positive + false_negative, 1)
    )
    f1 = (
        2 * precision * recall
        / max(precision + recall, 1e-12)
    )
    identity_accuracy = (
        exact_identity
        / max(true_positive, 1)
    )
    none_accuracy = (
        none_correct
        / max(none_count, 1)
    )

    metrics = {
        "n_examples": len(indices),
        "loss": loss_sum / max(loss_examples, 1),
        "P": precision,
        "R": recall,
        "F1": f1,
        "identity_accuracy_among_detected_starts": (
            identity_accuracy
        ),
        "NONE_accuracy": none_accuracy,
        "true_starts": true_start_count,
        "detected_gold_starts": true_positive,
        "false_triggers": false_positive,
        "missed_starts": false_negative,
        "exact_entity_identity": exact_identity,
        "none_positions": none_count,
    }

    if verbose:
        print(
            "selector validation | "
            f"loss={metrics['loss']:.4f} "
            f"P={precision:.4f} "
            f"R={recall:.4f} "
            f"F1={f1:.4f} "
            f"identity={identity_accuracy:.4f} "
            f"NONE={none_accuracy:.4f}"
        )

    return metrics


selector = EntitySelector().to(DEVICE)
optimizer = torch.optim.AdamW(
    selector.parameters(),
    lr=SELECTOR_LR,
)

start_epoch = 0
history = []
best_f1 = -1.0
best_identity = -1.0
best_epoch = 0
epochs_without_improvement = 0
early_stopped = False

if os.path.exists(SEL_LAST_PATH):
    resume_checkpoint = torch.load(
        SEL_LAST_PATH,
        map_location=DEVICE,
    )
    selector.load_state_dict(
        resume_checkpoint["state"]
    )
    optimizer.load_state_dict(
        resume_checkpoint["optimizer"]
    )
    start_epoch = int(
        resume_checkpoint.get("epoch", 0)
    )
    history = list(
        resume_checkpoint.get("history", [])
    )
    best_f1 = float(
        resume_checkpoint.get("best_f1", -1.0)
    )
    best_identity = float(
        resume_checkpoint.get(
            "best_identity",
            -1.0,
        )
    )
    best_epoch = int(
        resume_checkpoint.get("best_epoch", 0)
    )
    epochs_without_improvement = int(
        resume_checkpoint.get(
            "epochs_without_improvement",
            0,
        )
    )
    early_stopped = bool(
        resume_checkpoint.get(
            "early_stopped",
            False,
        )
    )
    print(
        "Resumed v5 selector at epoch",
        start_epoch,
        "| best epoch",
        best_epoch,
        "| best F1",
        best_f1,
    )
else:
    print(
        "Starting a fresh v5 selector run. "
        "Old v4 selector checkpoints are intentionally not loaded."
    )


def is_validation_improvement(
    metrics,
    previous_best_f1,
    previous_best_identity,
):
    current_f1 = float(metrics["F1"])
    current_identity = float(
        metrics[
            "identity_accuracy_among_detected_starts"
        ]
    )

    if current_f1 > previous_best_f1 + MIN_DELTA:
        return True

    f1_tied = (
        abs(current_f1 - previous_best_f1)
        <= MIN_DELTA
    )
    identity_improved = (
        current_identity
        > previous_best_identity + MIN_DELTA
    )
    return f1_tied and identity_improved


if early_stopped:
    print(
        "The saved v5 run already reached early stopping. "
        "Training is skipped."
    )

elif start_epoch >= MAX_SELECTOR_EPOCHS:
    print(
        "The saved v5 run already reached "
        f"MAX_SELECTOR_EPOCHS={MAX_SELECTOR_EPOCHS}."
    )

else:
    training_indices = list(
        range(len(data["train"]))
    )

    for epoch in range(
        start_epoch,
        MAX_SELECTOR_EPOCHS,
    ):
        selector.train()
        random.Random(
            SEED + epoch
        ).shuffle(training_indices)

        epoch_loss_sum = 0.0
        update_count = 0
        epoch_start_time = time.time()

        for batch_start in range(
            0,
            len(training_indices),
            SELECTOR_BATCH_SIZE,
        ):
            batch_indices = training_indices[
                batch_start:
                batch_start + SELECTOR_BATCH_SIZE
            ]
            batch_examples = [
                data["train"][index]
                for index in batch_indices
            ]
            batch_graphs = [
                graphs["train"][index]
                for index in batch_indices
            ]

            (
                fused_states,
                all_entity_embeddings,
                graph_assignment,
                target_ids,
            ) = teacher_states(
                batch_examples,
                batch_graphs,
            )

            batch_loss = None
            usable_examples = 0

            for local_index, (
                example,
                graph,
            ) in enumerate(
                zip(batch_examples, batch_graphs)
            ):
                entity_names = list(
                    getattr(
                        graph,
                        "entity_names",
                        [],
                    )
                    or []
                )
                if not entity_names:
                    continue

                node_indices = (
                    graph_assignment == local_index
                ).nonzero(
                    as_tuple=False
                ).flatten()

                if len(node_indices) != len(entity_names):
                    raise RuntimeError(
                        "Graph node/entity-name mismatch "
                        f"for train index "
                        f"{batch_indices[local_index]}: "
                        f"{len(node_indices)} vs "
                        f"{len(entity_names)}."
                    )

                entity_embeddings = (
                    all_entity_embeddings[
                        node_indices
                    ].float()
                )

                target_row = target_ids[local_index]
                target_length = int(
                    (
                        target_row != pad_id
                    ).sum().item()
                )
                if target_length < 2:
                    continue

                gold_labels = torch.from_numpy(
                    target_labels(
                        example,
                        graph,
                        target_row[
                            :target_length
                        ].tolist(),
                    )
                ).long().to(DEVICE)

                fused_row = fused_states[
                    local_index,
                    :target_length,
                ].float()

                example_loss = selector_loss_for_example(
                    selector,
                    fused_row,
                    entity_embeddings,
                    gold_labels,
                )

                if torch.isfinite(example_loss):
                    batch_loss = (
                        example_loss
                        if batch_loss is None
                        else batch_loss + example_loss
                    )
                    usable_examples += 1

            if usable_examples == 0:
                continue

            batch_loss = batch_loss / usable_examples

            optimizer.zero_grad(
                set_to_none=True
            )
            batch_loss.backward()
            torch.nn.utils.clip_grad_norm_(
                selector.parameters(),
                max_norm=1.0,
            )
            optimizer.step()

            epoch_loss_sum += float(
                batch_loss.detach().cpu()
            )
            update_count += 1

            if update_count % 50 == 0:
                print(
                    f"epoch {epoch + 1} "
                    f"step {update_count}: "
                    f"train loss "
                    f"{epoch_loss_sum / update_count:.4f} "
                    f"({time.time() - epoch_start_time:.0f}s)"
                )

        mean_train_loss = (
            epoch_loss_sum
            / max(update_count, 1)
        )

        validation_metrics = evaluate_selector(
            selector,
            SELECTOR_VAL_IDX,
            batch_size=16,
            verbose=True,
        )

        improved = is_validation_improvement(
            validation_metrics,
            best_f1,
            best_identity,
        )

        if improved:
            best_f1 = float(
                validation_metrics["F1"]
            )
            best_identity = float(
                validation_metrics[
                    "identity_accuracy_among_detected_starts"
                ]
            )
            best_epoch = epoch + 1
            epochs_without_improvement = 0

            atomic_torch_save(
                {
                    "state": selector.state_dict(),
                    "epoch": best_epoch,
                    "validation": validation_metrics,
                    "best_f1": best_f1,
                    "best_identity": best_identity,
                    "split_sha256": DEV_SPLIT_SHA256,
                    "config": {
                        "max_epochs": MAX_SELECTOR_EPOCHS,
                        "patience": PATIENCE,
                        "min_delta": MIN_DELTA,
                        "batch_size": SELECTOR_BATCH_SIZE,
                        "selector_lr": SELECTOR_LR,
                        "none_weight": NONE_WEIGHT,
                    },
                },
                SEL_BEST_PATH,
            )
            print(
                "NEW BEST selector checkpoint:",
                f"epoch={best_epoch}",
                f"F1={best_f1:.4f}",
                f"identity={best_identity:.4f}",
            )
        else:
            epochs_without_improvement += 1
            print(
                "No validation improvement. "
                f"patience={epochs_without_improvement}/{PATIENCE}"
            )

        history_entry = {
            "epoch": epoch + 1,
            "mean_train_loss": mean_train_loss,
            "updates": update_count,
            "elapsed_sec": (
                time.time() - epoch_start_time
            ),
            "validation": validation_metrics,
            "is_best": improved,
            "best_epoch_after_this_epoch": best_epoch,
            "epochs_without_improvement": (
                epochs_without_improvement
            ),
        }
        history.append(history_entry)

        will_stop = (
            epochs_without_improvement >= PATIENCE
        )

        atomic_torch_save(
            {
                "state": selector.state_dict(),
                "optimizer": optimizer.state_dict(),
                "epoch": epoch + 1,
                "history": history,
                "best_f1": best_f1,
                "best_identity": best_identity,
                "best_epoch": best_epoch,
                "epochs_without_improvement": (
                    epochs_without_improvement
                ),
                "early_stopped": will_stop,
                "split_sha256": DEV_SPLIT_SHA256,
            },
            SEL_LAST_PATH,
        )
        atomic_json_dump(
            history,
            SEL_HISTORY_PATH,
        )

        print(
            f"epoch {epoch + 1} complete | "
            f"train={mean_train_loss:.4f} | "
            f"val_F1={validation_metrics['F1']:.4f} | "
            f"best_epoch={best_epoch}"
        )

        if will_stop:
            early_stopped = True
            print(
                f"Early stopping after epoch {epoch + 1}: "
                f"no improvement for {PATIENCE} epochs."
            )
            break


if not os.path.exists(SEL_BEST_PATH):
    raise FileNotFoundError(
        "No best selector checkpoint was produced: "
        + SEL_BEST_PATH
    )

best_checkpoint = torch.load(
    SEL_BEST_PATH,
    map_location=DEVICE,
)
assert (
    best_checkpoint.get("split_sha256")
    == DEV_SPLIT_SHA256
), "Best checkpoint was trained with a different dev split manifest."

selector.load_state_dict(
    best_checkpoint["state"]
)
selector.eval()

for parameter in selector.parameters():
    parameter.requires_grad_(False)

BEST_SELECTOR_EPOCH = int(
    best_checkpoint["epoch"]
)
BEST_SELECTOR_SHA256 = sha256_file(
    SEL_BEST_PATH
)

print("\nBEST SELECTOR RESTORED")
print("epoch:", BEST_SELECTOR_EPOCH)
print(
    "validation:",
    best_checkpoint["validation"],
)
print("checkpoint:", SEL_BEST_PATH)
print("checkpoint SHA256:", BEST_SELECTOR_SHA256)
print(
    "selector parameters:",
    f"{sum(p.numel() for p in selector.parameters()):,}",
)

In [ ]:
# 7. Final intrinsic report for the restored best selector.
assert (
    "teacher_states" in globals()
    and callable(teacher_states)
), "Run Section 6 first."
assert "selector" in globals(), "Run Section 6 first."
assert os.path.exists(SEL_BEST_PATH)

selector.eval()

SEL_VAL = evaluate_selector(
    selector,
    SELECTOR_VAL_IDX,
    batch_size=16,
    verbose=True,
)
SEL_VAL.update({
    "best_epoch": BEST_SELECTOR_EPOCH,
    "selector_checkpoint": SEL_BEST_PATH,
    "selector_sha256": BEST_SELECTOR_SHA256,
    "split_manifest_sha256": DEV_SPLIT_SHA256,
})

SELECTOR_VALIDATION_PATH = os.path.join(
    EVAL_OUT,
    "selector_validation_best_v5.json",
)
atomic_json_dump(
    SEL_VAL,
    SELECTOR_VALIDATION_PATH,
)

if SEL_VAL["R"] < 0.2:
    print(
        "WARNING: selector rarely fires and may have "
        "collapsed toward NONE."
    )

print(
    "Saved best-selector validation:",
    SELECTOR_VALIDATION_PATH,
)

In [ ]:
# 8. Decode: selector trigger with exact-sequence commitment; hard_v2 backstop optional.
@torch.no_grad()
def decode_selector(ex, graph, tau=0.9, margin=6.0, use_selector=True, use_hard=True):
    enc_in = tokenizer(ex['linearized'], max_length=MAX_INPUT_LEN, truncation=True,
                       padding='max_length', return_tensors='pt')
    inp, att = enc_in['input_ids'].to(DEVICE), enc_in['attention_mask'].to(DEVICE)
    gb = torch.zeros(graph.x.size(0), dtype=torch.long, device=DEVICE)
    enc = model.bart.model.encoder(input_ids=inp, attention_mask=att)
    h_kg, _ = model.rgcn(graph.x.to(DEVICE), graph.edge_index.to(DEVICE), graph.edge_type.to(DEVICE))
    h_pad, k_mask = pad_kg_nodes(h_kg, gb, 1)
    names = list(getattr(graph, 'entity_names', []) or [])
    trie = TrieMapV2(names, tokenizer)
    ents = h_kg.float()
    ent_ids = {}
    for ni, base in trie.kept.items():
        b = tokenizer.encode(base, add_special_tokens=False)
        m_ = tokenizer.encode(' ' + base, add_special_tokens=False)
        if b and m_: ent_ids[ni] = (b, m_)
    dec_start, eos = model.bart.config.decoder_start_token_id, model.bart.config.eos_token_id
    gen = [dec_start]; past = None
    committed = None   # {'ids': [...], 'pos': int} — the SELECTED entity's exact sequence (v2 fix)
    triggers = 0
    for _ in range(MAX_GEN_LEN - 1):
        di = torch.tensor([[gen[-1]]], dtype=torch.long, device=DEVICE)
        out = model.bart.model.decoder(input_ids=di, encoder_hidden_states=enc.last_hidden_state,
                                       encoder_attention_mask=att, past_key_values=past, use_cache=True)
        past = out.past_key_values
        h, _, _ = model.kg_cross_attention(out.last_hidden_state, h_pad, k_mask)
        logits = add_final_logits_bias(model.bart, model.bart.lm_head(h))[:, -1, :].squeeze(0).float().cpu()
        if committed is not None:
            nxt = committed['ids'][committed['pos']]
            committed['pos'] += 1
            if committed['pos'] >= len(committed['ids']): committed = None
        else:
            logits2 = hard_mask_v2(logits, trie, gen[1:])[0] if use_hard else logits
            nxt = int(logits2.argmax())
            if use_selector and ent_ids:
                decoded = tokenizer.decode(gen[1:], skip_special_tokens=True)
                dn = _norm(decoded)
                cov_flags = [1.0 if (trie.kept.get(ni) and _wm(dn, _norm(trie.kept[ni]))) else 0.0
                             for ni in range(len(names))]
                cov = torch.tensor([cov_flags], device=DEVICE)
                sel = selector(h[:, -1, :].float().to(DEVICE), ents, cov).squeeze(0)
                ent_logits = sel[1:].clone()
                for ni in range(len(names)):          # v2 fix: mask covered/ineligible BEFORE argmax
                    if ni not in ent_ids or cov_flags[ni] > 0: ent_logits[ni] = -1e9
                probs = torch.softmax(torch.cat([sel[:1], ent_logits]), -1)
                if probs.numel() > 1:
                    best_ni = int(probs[1:].argmax()); p_best = float(probs[1 + best_ni])
                    if p_best >= tau and ent_ids.get(best_ni):
                        seq = ent_ids[best_ni][0] if len(gen) == 1 else ent_ids[best_ni][1]
                        if float(logits.max()) - float(logits[seq[0]]) <= margin:
                            nxt = seq[0]; triggers += 1
                            if len(seq) > 1: committed = {'ids': seq, 'pos': 1}
        if nxt == eos: break
        gen.append(nxt)
    return tokenizer.decode(gen[1:], skip_special_tokens=True), triggers


# REFERENCE RESOLUTION AND INTEGRITY
REFERENCE_FILES = (
    'preds_fusion_off.json',
    'preds_fusion_hard_v2.json',
)

def _valid_reference_dir(directory):
    return (
        directory
        and os.path.isdir(directory)
        and all(
            os.path.isfile(os.path.join(directory, filename))
            for filename in REFERENCE_FILES
        )
    )

def find_reference_directory():
    preferred = [
        HV2_EVAL,
        f'{PROJECT_DIR}/hard_v2_eval_fixed',
        f'{PROJECT_DIR}/hard_v2_eval',
        f'{PROJECT_DIR}/hard_v2_eval_fixed_v2',
    ]
    for directory in preferred:
        if _valid_reference_dir(directory):
            return directory

    print('Preferred reference folders not found; searching PROJECT_DIR recursively...')
    for root, _, files in os.walk(PROJECT_DIR):
        file_set = set(files)
        if all(filename in file_set for filename in REFERENCE_FILES):
            return root
    return None

REFERENCE_DIR = find_reference_directory()
REFERENCE_SOURCE = None
REFERENCE_BYTE_VERIFIED = False
REFERENCE_PROBE_RESULTS = {}

if REFERENCE_DIR is not None:
    print('Found saved 12b references:', REFERENCE_DIR)
    ref_off_probe = json.load(
        open(os.path.join(REFERENCE_DIR, 'preds_fusion_off.json'))
    )
    ref_hv2_probe = json.load(
        open(os.path.join(REFERENCE_DIR, 'preds_fusion_hard_v2.json'))
    )

    assert len(ref_off_probe) == len(ITEMS), (
        f'fusion_off length mismatch: {len(ref_off_probe)} != {len(ITEMS)}'
    )
    assert len(ref_hv2_probe) == len(ITEMS), (
        f'fusion_hard_v2 length mismatch: {len(ref_hv2_probe)} != {len(ITEMS)}'
    )

    for probe_index in (0, 7, 100, 500, 1500):
        generated_off, _ = decode_selector(
            ITEMS[probe_index]['ex'],
            graphs['test'][ITEMS[probe_index]['idx']],
            use_selector=False,
            use_hard=False,
        )
        generated_hv2, _ = decode_selector(
            ITEMS[probe_index]['ex'],
            graphs['test'][ITEMS[probe_index]['idx']],
            use_selector=False,
            use_hard=True,
        )

        off_match = generated_off == ref_off_probe[probe_index]
        hv2_match = generated_hv2 == ref_hv2_probe[probe_index]
        REFERENCE_PROBE_RESULTS[str(probe_index)] = {
            'fusion_off_match': off_match,
            'fusion_hard_v2_match': hv2_match,
        }

        assert off_match, f'fusion_off integrity FAIL at {probe_index}'
        assert hv2_match, f'fusion_hard_v2 integrity FAIL at {probe_index}'

    REFERENCE_SOURCE = 'saved_notebook_12b'
    REFERENCE_BYTE_VERIFIED = True
    print('Integrity passed against saved 12b outputs.')

else:
    print('\nWARNING: saved Notebook 12b predictions were not found.')
    for filename in REFERENCE_FILES:
        print('missing:', filename)

    if not ALLOW_REFERENCE_REGEN:
        raise FileNotFoundError(
            'Extract/copy both prediction files under PROJECT_DIR, '
            'or set ALLOW_REFERENCE_REGEN=True.'
        )

    REFERENCE_SOURCE = 'regenerated_same_checkpoint'
    REFERENCE_BYTE_VERIFIED = False

    # Smoke-test both no-selector paths. Historical byte identity cannot be
    # claimed because the historical files are absent.
    for probe_index in (0, 7, 100, 500, 1500):
        generated_off, _ = decode_selector(
            ITEMS[probe_index]['ex'],
            graphs['test'][ITEMS[probe_index]['idx']],
            use_selector=False,
            use_hard=False,
        )
        generated_hv2, _ = decode_selector(
            ITEMS[probe_index]['ex'],
            graphs['test'][ITEMS[probe_index]['idx']],
            use_selector=False,
            use_hard=True,
        )
        REFERENCE_PROBE_RESULTS[str(probe_index)] = {
            'fusion_off_generated': generated_off,
            'fusion_hard_v2_generated': generated_hv2,
        }

    print(
        'Fallback enabled. Full reference predictions will be regenerated '
        'resume-safely only if the selector passes both dev screens.'
    )

reference_metadata = {
    'source': REFERENCE_SOURCE,
    'reference_directory': REFERENCE_DIR,
    'byte_verified_against_saved_12b': REFERENCE_BYTE_VERIFIED,
    'allow_reference_regeneration': ALLOW_REFERENCE_REGEN,
    'probe_results': REFERENCE_PROBE_RESULTS,
}

REFERENCE_METADATA_PATH = os.path.join(
    EVAL_OUT,
    'reference_metadata_v5.json',
)
with open(REFERENCE_METADATA_PATH, 'w') as handle:
    json.dump(reference_metadata, handle, indent=2, ensure_ascii=False)

print('Reference mode:', REFERENCE_SOURCE)
print('Saved metadata:', REFERENCE_METADATA_PATH)

In [ ]:
# 9. New two-stage development screening using the restored best selector.
#
# Primary comparison:
#   fusion + adaptive selector commitment
#   versus
#   fusion off
#
# Continuation gate:
#   recall gain >= +0.5 percentage points on BOTH tune and confirmation.
#
# Final test success bar remains stricter:
#   recall gain >= +2.0 percentage points.

TUNE_PATH = os.path.join(
    EVAL_OUT,
    "selector_v5_frozen_config.json",
)


def dev_items_from_indices(indices):
    items = []
    for index in indices:
        example = dict(data["dev"][index])
        references = (
            example.get("all_targets")
            or [example.get("target", "")]
        )
        example["all_targets"] = [
            reference
            for reference in references
            if str(reference).strip()
        ]
        items.append({
            "idx": index,
            "ex": example,
        })
    return items


def eval_arm(
    items,
    tau,
    margin,
    use_selector,
    use_hard,
):
    predictions = []
    trigger_counts = []

    for item in items:
        prediction, triggers = decode_selector(
            item["ex"],
            graphs["dev"][item["idx"]],
            tau=tau,
            margin=margin,
            use_selector=use_selector,
            use_hard=use_hard,
        )
        predictions.append(prediction)
        trigger_counts.append(triggers)

    grounding = [
        grounding_score(
            prediction,
            item["ex"]["triples"],
        )
        for prediction, item in zip(
            predictions,
            items,
        )
    ]

    return {
        "n": len(items),
        "bleu": corpus_bleu_lc(
            predictions,
            [
                item["ex"]["all_targets"]
                for item in items
            ],
        ),
        "halluc": float(
            np.mean([
                score["halluc"]
                for score in grounding
            ])
        ),
        "recall": float(
            np.mean([
                score["recall"]
                for score in grounding
            ])
        ),
        "art": int(sum(
            artrow(prediction)
            for prediction in predictions
        )),
        "total_triggers": int(sum(trigger_counts)),
        "examples_triggered": int(sum(
            count > 0
            for count in trigger_counts
        )),
    }


def passes_screen(
    result,
    baseline,
    recall_gain,
):
    return (
        result["recall"]
        >= baseline["recall"] + recall_gain
        and baseline["bleu"] - result["bleu"]
        <= 0.7
        and result["halluc"]
        <= baseline["halluc"] + 0.003
        and result["art"]
        <= baseline["art"] + 3
    )


def config_matches_current_selector(config):
    return (
        config.get("selector_sha256")
        == BEST_SELECTOR_SHA256
        and config.get("split_manifest_sha256")
        == DEV_SPLIT_SHA256
        and config.get("best_selector_epoch")
        == BEST_SELECTOR_EPOCH
    )


if os.path.exists(TUNE_PATH):
    existing_config = json.load(
        open(TUNE_PATH, encoding="utf-8")
    )

    if not config_matches_current_selector(
        existing_config
    ):
        stale_path = (
            TUNE_PATH
            + ".stale_"
            + str(int(time.time()))
        )
        os.replace(TUNE_PATH, stale_path)
        print(
            "Archived stale tuning config:",
            stale_path,
        )


if not os.path.exists(TUNE_PATH):
    tune_items = dev_items_from_indices(
        V5_TUNE_IDX
    )
    confirm_items = dev_items_from_indices(
        V5_CONFIRM_IDX
    )

    tune_baseline = eval_arm(
        tune_items,
        tau=9.9,
        margin=0.0,
        use_selector=False,
        use_hard=False,
    )
    print(
        "NEW TUNE reference fusion_off:",
        tune_baseline,
    )

    tune_results = {}
    for tau in (0.70, 0.85, 0.95):
        for margin in (4.0, 8.0):
            key = f"t{tau}_m{margin}"
            result = eval_arm(
                tune_items,
                tau=tau,
                margin=margin,
                use_selector=True,
                use_hard=False,
            )
            tune_results[key] = result
            print(
                f"tau={tau} margin={margin} "
                "selector-only:",
                result,
            )

    passing_tune = {
        key: result
        for key, result in tune_results.items()
        if passes_screen(
            result,
            tune_baseline,
            recall_gain=0.005,
        )
    }

    chosen = None

    if passing_tune:
        best_key = max(
            passing_tune,
            key=lambda key: (
                passing_tune[key]["recall"],
                -passing_tune[key]["halluc"],
                passing_tune[key]["bleu"],
            ),
        )

        tau = float(
            best_key.split("_")[0][1:]
        )
        margin = float(
            best_key.split("_")[1][1:]
        )

        confirm_baseline = eval_arm(
            confirm_items,
            tau=9.9,
            margin=0.0,
            use_selector=False,
            use_hard=False,
        )
        confirm_result = eval_arm(
            confirm_items,
            tau=tau,
            margin=margin,
            use_selector=True,
            use_hard=False,
        )

        print(
            "NEW CONFIRM baseline:",
            confirm_baseline,
        )
        print(
            "NEW CONFIRM selector-only:",
            confirm_result,
        )

        if passes_screen(
            confirm_result,
            confirm_baseline,
            recall_gain=0.005,
        ):
            chosen = {
                "passed": True,
                "tau": tau,
                "margin": margin,
                "selected_key": best_key,
                "tune": passing_tune[best_key],
                "tune_base": tune_baseline,
                "confirm": confirm_result,
                "confirm_base": confirm_baseline,
            }

    if chosen is None:
        chosen = {
            "passed": False,
            "tune_results": tune_results,
            "tune_base": tune_baseline,
            "reason": (
                "No configuration passed both new "
                "development screens."
            ),
        }

    chosen.update({
        "selector_sha256": BEST_SELECTOR_SHA256,
        "best_selector_epoch": BEST_SELECTOR_EPOCH,
        "split_manifest_sha256": DEV_SPLIT_SHA256,
        "selector_validation": SEL_VAL,
        "screening_protocol": {
            "tune_n": len(V5_TUNE_IDX),
            "confirm_n": len(V5_CONFIRM_IDX),
            "continuation_recall_gain": 0.005,
            "max_bleu_drop": 0.7,
            "max_hallucination_increase": 0.003,
            "max_artifact_row_increase": 3,
        },
    })

    atomic_json_dump(
        chosen,
        TUNE_PATH,
    )


CFG = json.load(
    open(TUNE_PATH, encoding="utf-8")
)
assert config_matches_current_selector(CFG)

print(
    "FROZEN:",
    {
        key: CFG.get(key)
        for key in (
            "tau",
            "margin",
            "passed",
            "best_selector_epoch",
        )
    },
)

if not CFG.get("passed"):
    print(
        "\nThe best selector failed the two-stage "
        "development continuation gate."
    )
    print(
        "The repaired test decode below is skipped "
        "automatically."
    )

In [ ]:
# 10. Test decode: BOTH arms with the frozen config; full 2x2 attribution table.
if CFG.get('passed'):
    def atomic_json_dump(value, path):
        temporary_path = path + '.tmp'
        with open(temporary_path, 'w') as handle:
            json.dump(value, handle, ensure_ascii=False)
        os.replace(temporary_path, path)

    def load_or_generate_reference(tag, use_hard):
        if REFERENCE_DIR is not None:
            saved_path = os.path.join(
                REFERENCE_DIR,
                f'preds_{tag}.json',
            )
            predictions = json.load(open(saved_path))
            assert len(predictions) == len(ITEMS), (
                f'{tag} length mismatch: {len(predictions)} != {len(ITEMS)}'
            )
            return predictions

        if not ALLOW_REFERENCE_REGEN:
            raise FileNotFoundError(f'No reference predictions for {tag}.')

        cache_dir = os.path.join(
            EVAL_OUT,
            'regenerated_references',
        )
        os.makedirs(cache_dir, exist_ok=True)
        path = os.path.join(cache_dir, f'preds_{tag}.json')

        predictions = json.load(open(path)) if os.path.exists(path) else []
        assert len(predictions) <= len(ITEMS)

        started = time.time()
        for index in range(len(predictions), len(ITEMS)):
            prediction, trigger_count = decode_selector(
                ITEMS[index]['ex'],
                graphs['test'][ITEMS[index]['idx']],
                use_selector=False,
                use_hard=use_hard,
            )
            assert trigger_count == 0
            predictions.append(prediction)

            if (index + 1) % 25 == 0 or (index + 1) == len(ITEMS):
                atomic_json_dump(predictions, path)

            if (index + 1) % 250 == 0:
                print(
                    f'[reference {tag}] {index + 1}/{len(ITEMS)} '
                    f'({time.time() - started:.0f}s)'
                )

        return predictions

    ref = {
        'fusion_off': load_or_generate_reference(
            'fusion_off',
            use_hard=False,
        ),
        'fusion_hard_v2': load_or_generate_reference(
            'fusion_hard_v2',
            use_hard=True,
        ),
    }

    def run_test(tag, use_hard):
        path = os.path.join(EVAL_OUT, f'preds_{tag}.json'); tpath = os.path.join(EVAL_OUT, f'triggers_{tag}.json')
        preds = json.load(open(path)) if os.path.exists(path) else []
        trigs = json.load(open(tpath)) if os.path.exists(tpath) else []
        t0 = time.time()
        for i in range(len(preds), len(ITEMS)):
            p, tr = decode_selector(ITEMS[i]['ex'], graphs['test'][ITEMS[i]['idx']],
                                    CFG['tau'], CFG['margin'], True, use_hard)
            preds.append(p); trigs.append(tr)
            if (i + 1) % 25 == 0 or (i + 1) == len(ITEMS):
                atomic_json_dump(preds, path)
                atomic_json_dump(trigs, tpath)
            if (i + 1) % 250 == 0: print(f'[{tag}] {i+1}/2510 ({time.time()-t0:.0f}s)')
        return preds, trigs
    sel_only, trig1 = run_test('fusion_selector_only', use_hard=False)
    sel_hard, trig2 = run_test('fusion_selector_hard_v2', use_hard=True)

    def block(predictions, indices):
        subset_predictions = [predictions[i] for i in indices]
        subset_items = [ITEMS[i] for i in indices]
        scores = [
            grounding_score(p, item['ex']['triples'])
            for p, item in zip(subset_predictions, subset_items)
        ]
        return {
            'n': len(indices),
            'bleu': corpus_bleu_lc(
                subset_predictions,
                [item['ex']['all_targets'] for item in subset_items],
            ),
            'halluc': float(np.mean([g['halluc'] for g in scores])),
            'recall': float(np.mean([g['recall'] for g in scores])),
            'corr_rows': int(sum(len(g['corruptions']) > 0 for g in scores)),
            'art_rows': int(sum(artrow(p) for p in subset_predictions)),
        }

    all_indices = list(range(len(ITEMS)))
    seen_indices = [i for i, item in enumerate(ITEMS) if not item['unseen']]
    unseen_indices = [i for i, item in enumerate(ITEMS) if item['unseen']]

    arms = {
        'fusion_off': ref['fusion_off'],
        'fusion_selector_only': sel_only,
        'fusion_hard_v2': ref['fusion_hard_v2'],
        'fusion_selector_hard_v2': sel_hard,
    }

    out = {
        'reference_info': {
            'source': REFERENCE_SOURCE,
            'reference_directory': REFERENCE_DIR,
            'byte_verified_against_saved_12b': REFERENCE_BYTE_VERIFIED,
        },
        'selector_info': {
            'best_epoch': BEST_SELECTOR_EPOCH,
            'checkpoint': SEL_BEST_PATH,
            'checkpoint_sha256': BEST_SELECTOR_SHA256,
            'split_manifest_sha256': DEV_SPLIT_SHA256,
            'intrinsic_validation': SEL_VAL,
            'frozen_tau': CFG['tau'],
            'frozen_margin': CFG['margin'],
        },
        'overall': {
            name: block(predictions, all_indices)
            for name, predictions in arms.items()
        },
        'seen': {
            name: block(predictions, seen_indices)
            for name, predictions in arms.items()
        },
        'unseen': {
            name: block(predictions, unseen_indices)
            for name, predictions in arms.items()
        },
        'triggers': {
            'selector_only': int(sum(trig1)),
            'selector_hard': int(sum(trig2)),
            'examples_triggered_only': int(sum(1 for x in trig1 if x > 0)),
            'examples_triggered_hard': int(sum(1 for x in trig2 if x > 0)),
        },
    }

    summary_path = os.path.join(EVAL_OUT, 'selector_summary_v5.json')
    temporary_summary_path = summary_path + '.tmp'
    with open(temporary_summary_path, 'w') as summary_file:
        json.dump(out, summary_file, indent=2)
    os.replace(temporary_summary_path, summary_path)

    import pandas as pd
    display(pd.DataFrame([
        {'subset': subset, 'arm': arm, **metrics}
        for subset in ('overall', 'seen', 'unseen')
        for arm, metrics in out[subset].items()
    ]))
    print('Saved:', summary_path)
    print('\nFINAL success bars: selector_only vs fusion_off, and selector_hard_v2 vs fusion_hard_v2:')
    print('recall >= +2pp, BLEU drop <= 0.5, art_rows <= +10, corr_rows <= +5, halluc <= +0.2pp.')

In [ ]:
# 11. Package the complete v5 result directory.
zip_base = os.path.join(
    PROJECT_DIR,
    "selector_eval_v5_earlystop",
)
zip_path = shutil.make_archive(
    zip_base,
    "zip",
    EVAL_OUT,
)

print("Upload this single file:", zip_path)
print("\nFiles in result directory:")
for filename in sorted(os.listdir(EVAL_OUT)):
    print("  ", filename)